# FMAPIq N=200 test example

SAXS I(q) of a concentrated protein solution (cenA, 1959 atoms; N = 200 proteins in a 619.65 Å box,
0.016 M ions, 25 °C) from FMAP energy maps, NVT Monte Carlo and a Debye-type scattering calculation.

Run this notebook with a Julia kernel (`IJulia`) started from this directory, or open it in Google Colab:

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhougroup-uic/FMAPIqJ/blob/main/examples/n200_test/n200_test.ipynb)

On Colab pick a **GPU runtime** (Runtime → Change runtime type → T4); the first cell below clones the repository.
Switch `MODE` below: **`"gpu"`** = all stages on the GPU with 64 independent runs (about 4 min on an L40, about 10 min on a Colab T4),
**`"cpu"`** = CPU engines with 3 runs (much slower: the CPU energy maps alone take about 20 min).

> **Kernel:** locally, if Jupyter does not list a Julia kernel, run `julia tools/notebook.jl install` once
> (installs IJulia and a kernel `julia` started with `--project=<repo>`), or `julia tools/notebook.jl` to install and launch Jupyter.

## 0. Get the code (Colab / standalone notebook only)
Skipped when the notebook is run from inside a clone of the repository.

In [ ]:
# Colab / fresh machine: fetch the repository when the notebook was opened without it
let d = pwd(), found = false
    while true
        isfile(joinpath(d, "bin", "fmapiq.jl")) && (found = true; break)
        dirname(d) == d && break
        d = dirname(d)
    end
    if !found
        repo = joinpath(isdir("/content") ? "/content" : pwd(), "FMAPIqJ")
        isdir(repo) || run(`git -c advice.detachedHead=false clone --depth 1 --branch v0.1 https://github.com/zhougroup-uic/FMAPIqJ $repo`)
        cd(joinpath(repo, "examples", "n200_test"))
    end
    println("working directory: ", pwd())
end

## 1. Settings

In [ ]:
# "gpu" (64 runs) or "cpu" (3 runs); `nothing` = gpu if nvidia-smi sees a device, else cpu
MODE = nothing

has_gpu() = Sys.which("nvidia-smi") !== nothing && success(pipeline(`nvidia-smi -L`; stdout = devnull, stderr = devnull))
mode = something(MODE, has_gpu() ? "gpu" : "cpu")
mode in ("gpu", "cpu") || error("MODE must be \"gpu\" or \"cpu\"")
mode == "cpu" && haskey(ENV, "COLAB_RELEASE_TAG") &&
    @warn "no GPU visible: choose Runtime -> Change runtime type -> GPU, or CPU mode takes ~20 min for the energy maps"

NREPS    = mode == "gpu" ? 64 : 3          # independent runs
NCYCLES  = 20000                           # Monte Carlo cycles per run
THREADS  = 8                               # CPU threads (CPU mode / host work)

# repository root: the directory with bin/fmapiq.jl above the notebook
ROOT = let d = pwd()
    while !isfile(joinpath(d, "bin", "fmapiq.jl")) && dirname(d) != d; d = dirname(d); end
    d
end
WORK = joinpath(ROOT, "examples", "n200_test", "work_notebook_$(mode)")
println("mode = $mode, runs = $NREPS, cycles = $NCYCLES\nroot = $ROOT\nwork = $WORK")

## 2. Install packages (first time only)
Instantiates the project environment of the repository (downloads the Julia packages).

In [ ]:
using Pkg
Pkg.activate(ROOT)
Pkg.instantiate()

## 3. Run the pipeline
`bin/fmapiq.jl` runs the three stages in sequence: energy maps (`fmappre` → `fmap` → `fmapstat`),
NVT simulations, and P(q)/I(q) with statistics and plot. Output is streamed below.

In [ ]:
rm(WORK; recursive = true, force = true)
mkpath(joinpath(WORK, "cache"))
cp(joinpath(ROOT, "examples", "cenA.pqr"), joinpath(WORK, "subA.pqr"))

args = [joinpath(ROOT, "bin", "fmapiq.jl"), "subA.pqr", "--work-dir", "run",
        "--threads", string(THREADS), "--sim-threads", "2",
        "--n-reps", string(NREPS), "--n-cycles", string(NCYCLES),
        "--ion", "0.016", "--temp", "25", "--vscl", "0.16",
        "--n-proteins", "200", "--box-size", "619.65"]
mode == "gpu" && push!(args, "--gpu")
env = merge(ENV, Dict("FMAPIQ_CACHE" => joinpath(WORK, "cache")))
t = @elapsed run(setenv(`$(Base.julia_exename()) --project=$ROOT $args`, env; dir = WORK))
println("pipeline wall time: ", round(t / 60; digits = 2), " min")

## 4. Results

In [ ]:
using DelimitedFiles, Printf
RUN = joinpath(WORK, "run")
iqs  = readdlm(joinpath(RUN, "Iqs.txt"), Float64; comments = true)       # q, P(q), I_1/n ... I_R/n
stat = readdlm(joinpath(RUN, "Iqs.stat.txt"), Float64; comments = true)  # q, P/P0, mean/P0, SEM
@printf("P(0) = %.6e   (expected 2.186e6)\n", iqs[1, 2])
@printf("%d runs, %d q points, q = %.4f .. %.3f 1/A\n", size(iqs, 2) - 2, size(iqs, 1), iqs[2, 1], iqs[end, 1])

In [ ]:
# table of the first points: q, P(q)/P0, mean I(q)/P0 and its standard error over the runs
println("      q        P(q)/P0    I(q)/P0     SEM")
for i in (2, 5, 10, 20, 40, 80, 160, size(stat, 1))
    @printf("%8.4f  %10.5f  %10.5f  %10.5f\n", stat[i, 1], stat[i, 2], stat[i, 3], stat[i, 4])
end

The pipeline also writes `iq_plot.svg` (individual runs, mean ± SEM and the form factor P(q)):

In [ ]:
svg = read(joinpath(RUN, "iq_plot.svg"), String)
display("image/svg+xml", svg)

## 5. Check against the stored reference
`expected/Iqs.stat.n<runs>.txt` holds the result of the reference run with the same number of runs.
`tools/check_result.jl` passes if the mean curve agrees within 5 % or 3 standard errors.

In [ ]:
expected = joinpath(ROOT, "examples", "n200_test", "expected", "Iqs.stat.n$(NREPS).txt")
if NCYCLES == 20000 && isfile(expected)
    run(`$(Base.julia_exename()) --project=$ROOT $(joinpath(ROOT, "tools", "check_result.jl")) $(joinpath(RUN, "Iqs.stat.txt")) $expected 0.05`)
else
    println("no reference for NREPS = $NREPS, NCYCLES = $NCYCLES")
end